# TITAN Slide Embeddings for SurGen (CZI -> TITAN, one notebook)

This notebook produces TITAN slide-level embeddings directly from **SurGen CZI**
whole-slide images. Unlike `Titan_Project.ipynb` (which requires pre-cut PNG
patches on disk), this notebook reads patches straight out of the `.czi` files,
the same way `Complete_Pipeline/combined_pipeline_final_error_checks_v6.py`
does for its Step C feature extraction. It is fully self-contained: no other
repo script is imported.

## What it does differently from `Titan_Project.ipynb`

1. **Reads CZI, not PNG.** Patches are cut in memory via `pylibCZIrw` at
   512px @ 20x (TITAN's native patch geometry) and never written to disk.
2. **Tissue selection reuses existing work.** `patch_metadata_nonwhite.csv`
   (2+ GB) already records, per SurGen slide, every patch that survived the
   pixel-rule + SVM whiteness filter used by the CZI orchestrator. This
   notebook only *reads* that CSV (once, into a cached index) - it does not
   recompute RGB stats or refit any filter.
3. **Fixes a coordinate bug.** `Titan_Project.ipynb` halves 40x-slide
   coordinates into the 20x frame (`x // 2`) but then passes
   `patch_size_lv0 = 1024` to TITAN. TITAN's `preprocess_features` does
   `floor_divide(coords, patch_size_lv0)` and then `index_add_`s features
   into a grid - with halved coords and an un-halved patch size, every 2x2
   block of adjacent patches collapses into **one** grid cell and their
   features get summed together. This notebook emits true level-0
   coordinates (`patch_x * downsample`, `patch_y * downsample`) together with
   the matching `patch_size_lv0 = PATCH_SIZE * downsample`, and asserts that
   the resulting grid has exactly one patch per cell before encoding.
4. **Output path matches the existing SurGen aggregation layout**, i.e. the
   same tree `Averaging` and `Tissue_Type_Clustering` already write to:
   `surgen_processed/<model>/features/slide_aggregation/<METHOD>/<model>/<slide>.pt`.

## Environment

Run this notebook in the `TITAN_ENV` conda environment (the one that already
has `MahmoodLab/TITAN` cached locally). It is missing `pylibCZIrw`; install it
once with:

```
python -m pip install pylibCZIrw
```

`python-dotenv` is not required - `HF_TOKEN` is read from `os.environ`, with a
tiny inline `.env` parser as a fallback so you don't need the package.


In [ ]:
import os
import sys
import gc
import glob
import json
import time
import pickle
import shutil
import signal
import subprocess
import threading
import traceback
import faulthandler
import urllib.request
import urllib.error
from pathlib import Path
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm import tqdm


In [ ]:
# =============================================================================
# CONFIG - edit everything here
# =============================================================================

# -- Source data --------------------------------------------------------------
CZI_DIR               = r"F:\CZI_Files"
SURGEN_LABELS_CSV     = r"D:\Aamir Gulzar\KSA_project2\surgen_data\surgen_labels.csv"
NONWHITE_METADATA_CSV = r"D:\Aamir Gulzar\KSA_project2\surgen_data\patch_metadata_nonwhite.csv"

# -- Output (matches slide_classification/config/paths.py::feature_dir) -------
FEATURES_BASE  = r"D:\Aamir Gulzar\KSA_project2\surgen_data\surgen_processed"
MODEL_DIR_NAME = "conch1-5"          # TITAN's patch encoder is CoNCH v1.5
AGG_METHOD     = "TITAN"
OUTPUT_DIR = os.path.join(FEATURES_BASE, MODEL_DIR_NAME, "features",
                          "slide_aggregation", AGG_METHOD, MODEL_DIR_NAME)

# -- Patch geometry (must match how patch_metadata_nonwhite.csv was built) ----
PATCH_SIZE = 512      # px, at TARGET_MAG
TARGET_MAG = 20        # native TITAN patch resolution

# -- Coordinate handling --------------------------------------------------------
# "level0"  -> correct: emits true level-0 pixel coords + matching patch_size_lv0
#              (this is what TITAN's docstring specifies and what this notebook
#              defaults to)
# "legacy"  -> reproduces the Titan_Project.ipynb bug (halved coords, un-halved
#              patch_size_lv0) for direct comparison against old .pt files.
COORD_MODE = "level0"

# -- Runtime --------------------------------------------------------------------
PATCH_BATCH_SIZE = 32          # patches per CoNCH forward pass
DEVICE           = "cuda"

# -- Download / storage settings (only used if DOWNLOAD_ENABLED) ----------------
DOWNLOAD_ENABLED = True   # False = only process CZIs already sitting in CZI_DIR
DELETE_AFTER     = True   # delete each batch's CZIs once its slides are embedded
BATCH_SIZE       = 2       # WSIs per download/process batch
MIN_FREE_GB      = 20
DISK_TO_CHECK    = "F:\\"
ARIA2C_EXE = r"C:\Users\datainsight\AppData\Local\Microsoft\WinGet\Packages\aria2.aria2_Microsoft.Winget.Source_8wekyb3d8bbwe\aria2-1.37.0-win-64bit-build1\aria2c.exe"
ARIA2_RPC_PORT = 6800   # local JSON-RPC port aria2c reports live progress on
_BASE_URL  = "https://ftp.ebi.ac.uk/biostudies/fire/S-BIAD/285/S-BIAD1285/Files"

# -- Remote sync (disabled by default; same shape as the CZI orchestrator) ------
REMOTE_SYNC_ENABLED = False
REMOTE_USER = "mle"
REMOTE_HOST = "100.116.197.88"
REMOTE_BASE = "/media/dp-psau/Datum/Aamir/Azfaar/surgen_processed"
SSH_KEY     = r"C:\Users\datainsight\.ssh\id_ed25519"

# -- HuggingFace token ------------------------------------------------------------
def _load_dotenv_token():
    # Minimal .env reader for HF_TOKEN, so python-dotenv is not required.
    for env_path in (os.path.join(os.getcwd(), ".env"),
                     os.path.join(os.path.dirname(os.getcwd()), ".env")):
        if os.path.isfile(env_path):
            with open(env_path, "r") as f:
                for line in f:
                    line = line.strip()
                    if line.startswith("HF_TOKEN") and "=" in line:
                        return line.split("=", 1)[1].strip().strip('"').strip("'")
    return ""

HF_TOKEN = os.environ.get("HF_TOKEN", "") or _load_dotenv_token()

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"[CONFIG] Output dir : {OUTPUT_DIR}")
print(f"[CONFIG] Coord mode : {COORD_MODE}")
print(f"[CONFIG] HF token   : {'present' if HF_TOKEN else 'MISSING - set HF_TOKEN env var or a .env file'}")


In [ ]:
# =============================================================================
# CRASH DIAGNOSTICS (ported from combined_pipeline_final_error_checks_v6.py)
# pylibCZIrw can segfault natively; faulthandler + a breadcrumb file are the
# only way to know which slide/batch was in flight when that happens.
# =============================================================================

CRASH_DIAG_DIR = os.path.join(os.getcwd(), "crash_diagnostics")
os.makedirs(CRASH_DIAG_DIR, exist_ok=True)

_PID = os.getpid()
_FAULTHANDLER_LOG_PATH = os.path.join(CRASH_DIAG_DIR, f"faulthandler_{_PID}.log")
_LAST_STATE_PATH       = os.path.join(CRASH_DIAG_DIR, "last_state.json")
_BREADCRUMBS_PATH      = os.path.join(CRASH_DIAG_DIR, "breadcrumbs.log")

_faulthandler_fh = open(_FAULTHANDLER_LOG_PATH, "w", buffering=1)
faulthandler.enable(file=_faulthandler_fh, all_threads=True)
try:
    for _sig in (signal.SIGSEGV, signal.SIGABRT, signal.SIGILL, signal.SIGFPE):
        faulthandler.register(_sig, file=_faulthandler_fh, all_threads=True, chain=True)
except Exception:
    pass  # not all signals are registrable on every platform


def _now_iso():
    return datetime.now().isoformat(timespec="milliseconds")


def _get_gpu_mem_snapshot():
    try:
        if torch.cuda.is_available():
            return {
                "allocated_mb": round(torch.cuda.memory_allocated() / (1024 ** 2), 1),
                "reserved_mb":  round(torch.cuda.memory_reserved() / (1024 ** 2), 1),
            }
    except Exception:
        pass
    return None


def write_checkpoint(stage: str, **details):
    state = {"timestamp": _now_iso(), "pid": _PID, "stage": stage,
             "gpu_mem": _get_gpu_mem_snapshot(), "details": details}
    tmp_path = f"{_LAST_STATE_PATH}.{_PID}.{threading.get_ident()}.tmp"
    try:
        with open(tmp_path, "w") as f:
            json.dump(state, f, indent=2, default=str)
            f.flush()
            os.fsync(f.fileno())
        for attempt in range(5):
            try:
                os.replace(tmp_path, _LAST_STATE_PATH)
                break
            except PermissionError:
                if attempt == 4:
                    raise
                time.sleep(0.05 * (attempt + 1))
    except Exception as e:
        print(f"[DIAG WARN] Could not write last_state.json: {e}", flush=True)
        if os.path.exists(tmp_path):
            try:
                os.remove(tmp_path)
            except Exception:
                pass
    try:
        with open(_BREADCRUMBS_PATH, "a") as f:
            f.write(json.dumps(state, default=str) + "\n")
    except Exception as e:
        print(f"[DIAG WARN] Could not append breadcrumbs.log: {e}", flush=True)


def log_exception(context: str, exc: BaseException):
    tb_str = "".join(traceback.format_exception(type(exc), exc, exc.__traceback__))
    msg = f"\n[EXCEPTION @ {_now_iso()}] context={context}\n{tb_str}"
    print(msg, flush=True)
    try:
        with open(os.path.join(CRASH_DIAG_DIR, "exceptions.log"), "a") as f:
            f.write(msg + "\n")
    except Exception:
        pass


def log(msg: str):
    print(msg, flush=True)


write_checkpoint("process_start", python=sys.version)
print(f"[DIAG] faulthandler active -> {_FAULTHANDLER_LOG_PATH}")
print(f"[DIAG] breadcrumb state    -> {_LAST_STATE_PATH}")


In [ ]:
# =============================================================================
# WSI CATALOGUE + LABEL LOOKUP (ported from the CZI orchestrator)
# =============================================================================

def build_wsi_list(labels_csv: str) -> list:
    # Return (filename, url) pairs for every slide with label_desc != -1.
    df = pd.read_csv(labels_csv)
    if "WSI_Id" not in df.columns or "label_desc" not in df.columns:
        raise ValueError(f"Expected columns 'WSI_Id' and 'label_desc' in {labels_csv}. "
                         f"Found: {df.columns.tolist()}")

    valid_stems = df.loc[df["label_desc"] != -1, "WSI_Id"].tolist()
    wsis, skipped = [], 0
    for stem in valid_stems:
        stem_upper = stem.upper()
        if stem_upper.startswith("SR1482"):
            subfolder = "SR1482_WSIs"
        elif stem_upper.startswith("SR386"):
            subfolder = "SR386_WSIs"
        else:
            log(f"[WARN] Unrecognised prefix for '{stem}' - skipping.")
            skipped += 1
            continue
        filename = stem + ".czi"
        wsis.append((filename, f"{_BASE_URL}/{subfolder}/{filename}"))

    log(f"[CATALOGUE] {len(wsis)} valid WSI(s) loaded from {labels_csv} "
        f"({len(df) - len(valid_stems)} excluded with label=-1"
        + (f", {skipped} skipped due to unknown prefix" if skipped else "") + ")")
    return wsis


_surgen_labels_df = None


def _load_label_df():
    global _surgen_labels_df
    if _surgen_labels_df is None:
        df = pd.read_csv(SURGEN_LABELS_CSV)
        _surgen_labels_df = df.set_index("WSI_Id")


def get_slide_label(czi_filename: str):
    # Return 'msih' / 'nonmsih' / None for the given CZI filename.
    _load_label_df()
    stem = Path(czi_filename).stem
    if stem not in _surgen_labels_df.index:
        return None
    val = int(_surgen_labels_df.at[stem, "label_desc"])
    if val == 1:
        return "msih"
    if val == 0:
        return "nonmsih"
    return None


In [ ]:
# =============================================================================
# NON-WHITE PATCH COORDINATE INDEX
# patch_metadata_nonwhite.csv is 2+ GB - the RGB-stat + SVM whiteness filtering
# was already done when it was produced (Step A-1 / A-2 of the CZI
# orchestrator). This cell only reads slide_name/patch_x/patch_y from it once,
# builds a small per-slide coordinate index, and caches it to disk so re-runs
# (and the resume scan) never re-parse the CSV.
# =============================================================================

_NONWHITE_INDEX_CACHE = os.path.join(CRASH_DIAG_DIR, "nonwhite_index_cache.pkl")


def build_nonwhite_index(csv_path: str, cache_path: str) -> dict:
    # slide_name -> np.ndarray (N, 2) int32 of (patch_x, patch_y).
    csv_mtime = os.path.getmtime(csv_path)

    if os.path.exists(cache_path):
        with open(cache_path, "rb") as f:
            cached = pickle.load(f)
        if cached.get("csv_mtime") == csv_mtime and cached.get("csv_path") == csv_path:
            log(f"[INDEX] Loaded cached non-white index ({len(cached['index'])} slides) "
                f"from {cache_path}")
            return cached["index"]
        log("[INDEX] Cache stale (CSV changed) - rebuilding.")

    log(f"[INDEX] Building non-white coordinate index from {csv_path} ...")
    per_slide = {}
    t0 = time.time()
    reader = pd.read_csv(
        csv_path,
        usecols=["slide_name", "patch_x", "patch_y"],
        dtype={"patch_x": "int32", "patch_y": "int32"},
        chunksize=2_000_000,
    )
    n_rows = 0
    for chunk in reader:
        n_rows += len(chunk)
        for slide_name, group in chunk.groupby("slide_name", sort=False):
            arr = group[["patch_x", "patch_y"]].to_numpy(dtype=np.int32)
            if slide_name in per_slide:
                per_slide[slide_name] = np.concatenate([per_slide[slide_name], arr], axis=0)
            else:
                per_slide[slide_name] = arr
        log(f"  ... {n_rows:,} rows processed so far")

    log(f"[INDEX] Built index for {len(per_slide)} slide(s), {n_rows:,} rows, "
        f"in {time.time() - t0:.1f}s")

    with open(cache_path, "wb") as f:
        pickle.dump({"csv_path": csv_path, "csv_mtime": csv_mtime, "index": per_slide}, f)
    return per_slide


NONWHITE_INDEX = build_nonwhite_index(NONWHITE_METADATA_CSV, _NONWHITE_INDEX_CACHE)


def slide_has_tissue(stem: str) -> bool:
    return stem in NONWHITE_INDEX and len(NONWHITE_INDEX[stem]) > 0


def slide_done(stem: str) -> bool:
    return os.path.exists(os.path.join(OUTPUT_DIR, f"{stem}.pt"))


def slide_ready_to_skip(stem: str) -> bool:
    # True if there is nothing left to do for this slide: either it is
    # already embedded, or it is confirmed to have zero tissue patches.
    return slide_done(stem) or not slide_has_tissue(stem)


def filter_batch_for_download(batch: list) -> list:
    to_download = []
    for filename, url in batch:
        stem = Path(filename).stem
        if slide_done(stem):
            log(f"  [SKIP DOWNLOAD] {filename} - embedding already exists")
        elif not slide_has_tissue(stem):
            log(f"  [SKIP DOWNLOAD] {filename} - no non-white patches recorded, nothing to encode")
        else:
            to_download.append((filename, url))
    return to_download


def _batch_fully_complete(batch: list) -> bool:
    return all(slide_ready_to_skip(Path(filename).stem) for filename, _ in batch)


In [ ]:
# =============================================================================
# DOWNLOAD / DELETE / DISK GUARD / REMOTE SYNC (ported from the orchestrator)
# =============================================================================

def get_free_gb(path: str) -> float:
    return shutil.disk_usage(path).free / (1024 ** 3)


def check_disk_space_or_halt(threshold_gb: float = MIN_FREE_GB):
    free_gb = get_free_gb(DISK_TO_CHECK)
    log(f"[DISK CHECK] Free space on {DISK_TO_CHECK}: {free_gb:.1f} GB (threshold: {threshold_gb} GB)")
    if free_gb < threshold_gb:
        log(f"[HALT] Less than {threshold_gb} GB free on {DISK_TO_CHECK}. Stopping.")
        raise SystemExit(1)


def _aria2_rpc_call(port, method, params=None, timeout=5):
    """One JSON-RPC 2.0 call against aria2c's local RPC server (stdlib only)."""
    payload = {"jsonrpc": "2.0", "id": "titan-dl", "method": method, "params": params or []}
    req = urllib.request.Request(
        f"http://127.0.0.1:{port}/jsonrpc",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return json.loads(resp.read())


def _fmt_gb(n_bytes):
    return f"{n_bytes / (1024 ** 3):.2f} GB"


def _fmt_speed(bytes_per_sec):
    return f"{bytes_per_sec / (1024 ** 2):.1f} MB/s"


def _poll_aria2_progress(port, stop_event, poll_every=2.0):
    """Runs on a background thread while one aria2c download is in flight.

    aria2c's own live progress readout goes through the Windows Console API,
    which silently produces no output at all once stdout is a pipe instead of
    a real console (as it always is under a Jupyter subprocess) - streaming
    its stdout, or forcing --summary-interval, does not help, because aria2c
    itself never writes anything in that case. Its JSON-RPC interface is
    unaffected by that and gives exact completed/total bytes and speed
    regardless of console state, so progress is read from there instead.
    """
    last_gid = None
    while not stop_event.is_set():
        try:
            active = _aria2_rpc_call(port, "aria2.tellActive",
                                     [["gid", "completedLength", "totalLength", "downloadSpeed"]])
            results = active.get("result", [])
            if not results:
                # Nothing active yet (RPC server just started) or already finished.
                stop_event.wait(poll_every)
                continue
            r = results[0]
            last_gid = r.get("gid", last_gid)
            completed = int(r.get("completedLength", 0))
            total = int(r.get("totalLength", 0))
            speed = int(r.get("downloadSpeed", 0))
            pct = (completed / total * 100.0) if total else 0.0
            print(f"    {_fmt_gb(completed)} / {_fmt_gb(total)} ({pct:5.1f}%)  {_fmt_speed(speed)}",
                 flush=True)
        except (urllib.error.URLError, ConnectionError, OSError):
            pass  # RPC server not up yet, or download just finished and port closed
        except Exception as e:
            print(f"    [PROGRESS WARN] {type(e).__name__}: {e}", flush=True)
        stop_event.wait(poll_every)


def download_batch(batch: list):
    os.makedirs(CZI_DIR, exist_ok=True)
    log(f"[DOWNLOAD] Downloading {len(batch)} WSI(s) -> {CZI_DIR}")

    aria2c_resolved = shutil.which(ARIA2C_EXE) or (ARIA2C_EXE if os.path.isfile(ARIA2C_EXE) else None)
    if aria2c_resolved is None:
        log(f"[FATAL] aria2c not found at '{ARIA2C_EXE}'")
        raise SystemExit(1)

    for i, (filename, url) in enumerate(batch, 1):
        out_path = os.path.join(CZI_DIR, filename)
        log(f"  [{i}/{len(batch)}] {filename}")
        if os.path.exists(out_path):
            log(f"    [SKIP] Already exists ({_fmt_gb(os.path.getsize(out_path))})")
            continue

        cmd = [aria2c_resolved, "-x", "16", "-s", "16", "-c",
              "--enable-rpc=true", f"--rpc-listen-port={ARIA2_RPC_PORT}",
              "--rpc-listen-all=false",
              "-d", CZI_DIR, "-o", filename, url]
        t0 = time.time()
        stop_event = threading.Event()
        progress_thread = threading.Thread(
            target=_poll_aria2_progress, args=(ARIA2_RPC_PORT, stop_event), daemon=True
        )
        try:
            proc = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            progress_thread.start()
            returncode = proc.wait()
            elapsed = time.time() - t0
            if returncode == 0:
                log(f"    [OK] Downloaded in {elapsed:.1f}s -> {_fmt_gb(os.path.getsize(out_path)) if os.path.exists(out_path) else '?'}")
            else:
                log(f"    [ERROR] aria2c exited with code {returncode} after {elapsed:.1f}s")
        except Exception as e:
            log(f"    [ERROR] Unexpected error downloading {filename}: {type(e).__name__}: {e}")
        finally:
            stop_event.set()
            progress_thread.join(timeout=3)


def delete_batch_czis(batch: list):
    log(f"[CLEANUP] Deleting {len(batch)} CZI file(s) ...")
    for filename, _ in batch:
        path = os.path.join(CZI_DIR, filename)
        if os.path.exists(path):
            try:
                os.remove(path)
            except Exception as e:
                log(f"  [WARN] Could not delete {filename}: {e}")
        ctrl = path + ".aria2"
        if os.path.exists(ctrl):
            try:
                os.remove(ctrl)
            except Exception:
                pass


def _win_to_wsl(path: str) -> str:
    p = path.replace("\\", "/")
    return f"/mnt/{p[0].lower()}{p[2:]}"


def sync_to_remote(label: str = ""):
    if not REMOTE_SYNC_ENABLED:
        return
    log(f"[SYNC] Pushing {OUTPUT_DIR} to {REMOTE_USER}@{REMOTE_HOST}:{REMOTE_BASE} {label}")
    try:
        local_wsl = _win_to_wsl(FEATURES_BASE)
        ssh_opts = f"ssh -i {_win_to_wsl(SSH_KEY)} -o StrictHostKeyChecking=no -o BatchMode=yes"
        cmd = ["wsl", "rsync", "-avz", "--exclude=*.tmp", f"--rsh={ssh_opts}",
               local_wsl + "/", f"{REMOTE_USER}@{REMOTE_HOST}:{REMOTE_BASE}/"]
        result = subprocess.run(cmd, capture_output=False, timeout=3600)
        log("[SYNC OK]" if result.returncode == 0 else f"[SYNC ERROR] rc={result.returncode}")
    except Exception as e:
        log(f"[SYNC ERROR] {type(e).__name__}: {e}")


In [ ]:
# =============================================================================
# CZI READER (ported from the orchestrator)
#
# One persistent single-thread pool is reused for every slide's open/read/close
# calls. pylibCZIrw's native decoder keeps thread-affine state; recreating a
# ThreadPoolExecutor per slide (tearing down and respawning the OS thread it
# ran on) is the documented cause of intermittent access-violation crashes in
# this codebase. Reads stay fully serialized either way - only the OS thread
# identity is kept stable here.
# =============================================================================

_CZI_READER_POOL = None
_CZI_READER_POOL_LOCK = threading.Lock()


def _get_czi_reader_pool():
    global _CZI_READER_POOL
    with _CZI_READER_POOL_LOCK:
        if _CZI_READER_POOL is None:
            _CZI_READER_POOL = ThreadPoolExecutor(max_workers=1, thread_name_prefix="czi-reader")
    return _CZI_READER_POOL


def _shutdown_czi_reader_pool():
    global _CZI_READER_POOL
    with _CZI_READER_POOL_LOCK:
        if _CZI_READER_POOL is not None:
            try:
                _CZI_READER_POOL.shutdown(wait=True, cancel_futures=False)
            except Exception:
                pass
            _CZI_READER_POOL = None


import atexit
atexit.register(_shutdown_czi_reader_pool)


def open_czi_at_target_mag(czi_path: str, target_mag: int = 20, patch_size: int = 512):
    from pylibCZIrw import czi as pyczi

    write_checkpoint("czi_open_attempt", czi_path=czi_path)
    with pyczi.open_czi(czi_path) as czidoc:
        try:
            native_mag = int(
                czidoc.metadata["ImageDocument"]["Metadata"]["Information"]
                ["Instrument"]["Objectives"]["Objective"]["NominalMagnification"]
            )
        except Exception:
            native_mag = 40

        bbox = czidoc.total_bounding_box
        X_start, Y_start = bbox["X"][0], bbox["Y"][0]
        W, H = bbox["X"][1] - bbox["X"][0], bbox["Y"][1] - bbox["Y"][0]
        downsample = max(1, native_mag // target_mag)

    write_checkpoint("czi_open_success", czi_path=czi_path, W=W, H=H,
                      native_mag=native_mag, downsample=downsample)
    return {"czi_path": czi_path, "W": W, "H": H, "X_start": X_start, "Y_start": Y_start,
            "native_mag": native_mag, "downsample": downsample}


def _read_batch_sequential(czidoc, batch_rows, slide_stem, x_start, y_start, downsample,
                            size_native, zoom):
    # Read every patch in a batch sequentially on the calling thread.
    # pylibCZIrw's czidoc is not thread-safe for concurrent reads, so this must
    # only ever be called from the single dedicated reader thread.
    images, valid_rows = [], []
    for row in batch_rows:
        patch_x, patch_y = int(row[0]), int(row[1])
        x_native = x_start + patch_x * downsample
        y_native = y_start + patch_y * downsample
        try:
            raw = czidoc.read(roi=(x_native, y_native, size_native, size_native),
                              zoom=zoom, plane={"C": 0, "Z": 0, "T": 0})
            images.append(Image.fromarray(raw[:, :, :3].astype("uint8")).convert("RGB"))
            valid_rows.append(row)
        except Exception as e:
            log_exception(f"patch_read slide={slide_stem} x={patch_x} y={patch_y}", e)
    return images, valid_rows


In [ ]:
# =============================================================================
# TITAN EXTRACTOR
#
# TITAN's own patch encoder is CoNCH v1.5 (obtained via titan.return_conch()).
# Its eval_transform is Resize(448) -> CenterCrop(448) -> ToTensor ->
# Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)) - see conch_v1_5.py in the
# TITAN model repo. This is NOT the FiveCrop(256) transform the CZI
# orchestrator's Conch15Extractor uses for its own per-patch feature dumps:
# TITAN wants exactly one 768-d vector per 512@20x patch, not five.
# =============================================================================

from transformers import AutoModel
from huggingface_hub import login


class TITANExtractor:
    def __init__(self, token: str = HF_TOKEN, device: str = DEVICE):
        if token:
            login(token=token)
        log("Loading TITAN model...")
        self.titan = AutoModel.from_pretrained("MahmoodLab/TITAN", trust_remote_code=True)
        log("Model loaded. Initializing CONCH v1.5...")
        self.conch, self.eval_transform = self.titan.return_conch()

        self.device = torch.device(device if torch.cuda.is_available() else "cpu")
        log(f"Using device: {self.device}")
        self.conch = self.conch.to(self.device).eval()
        self.titan = self.titan.to(self.device).eval()

    def encode_patches(self, images) -> torch.Tensor:
        # list[PIL.Image, 512x512 RGB @20x] -> (N, 768) float32 cpu tensor.
        batch = torch.stack([self.eval_transform(im) for im in images]).to(self.device)
        with torch.autocast(device_type=self.device.type, dtype=torch.float16), torch.inference_mode():
            feats = self.conch(batch)
        return feats.float().cpu()


In [ ]:
# =============================================================================
# PER-SLIDE ENCODE
# =============================================================================

def _atomic_save(tensor: torch.Tensor, out_path: str):
    tmp_path = out_path + ".tmp"
    torch.save(tensor, tmp_path)
    os.replace(tmp_path, out_path)


def encode_slide(slide_stem: str, czi_path: str, extractor: "TITANExtractor",
                 coord_rows: np.ndarray) -> str:
    # Encode one slide's TITAN embedding and save it to OUTPUT_DIR/<stem>.pt.
    write_checkpoint("slide_start", slide=slide_stem, n_patches=len(coord_rows))
    t0 = time.time()

    slide_info = open_czi_at_target_mag(czi_path, TARGET_MAG, PATCH_SIZE)
    downsample = slide_info["downsample"]
    x0, y0 = slide_info["X_start"], slide_info["Y_start"]
    zoom = 1.0 / downsample
    size_native = PATCH_SIZE * downsample

    # Sort by (x, y) for locality and to match the reference notebook's convention.
    order = np.lexsort((coord_rows[:, 1], coord_rows[:, 0]))
    rows = coord_rows[order]

    reader_pool = _get_czi_reader_pool()
    from pylibCZIrw import czi as pyczi

    def _open_on_worker(path):
        ctx = pyczi.open_czi(path)
        return ctx, ctx.__enter__()

    czi_ctx, czidoc = reader_pool.submit(_open_on_worker, czi_path).result()

    all_feats, all_coords = [], []
    try:
        n_total = len(rows)
        n_batches = (n_total + PATCH_BATCH_SIZE - 1) // PATCH_BATCH_SIZE

        next_batch_rows = rows[0:PATCH_BATCH_SIZE]
        next_future = reader_pool.submit(_read_batch_sequential, czidoc, next_batch_rows,
                                         slide_stem, x0, y0, downsample, size_native, zoom)

        for b_idx in tqdm(range(n_batches), desc=f"  {slide_stem}", unit="batch", leave=False):
            images, valid_rows = next_future.result()

            if b_idx + 1 < n_batches:
                nb_rows = rows[(b_idx + 1) * PATCH_BATCH_SIZE: (b_idx + 2) * PATCH_BATCH_SIZE]
                next_future = reader_pool.submit(_read_batch_sequential, czidoc, nb_rows,
                                                 slide_stem, x0, y0, downsample, size_native, zoom)

            if not images:
                continue

            try:
                feats = extractor.encode_patches(images)
            except Exception as e:
                log_exception(f"inference slide={slide_stem} batch={b_idx}", e)
                continue

            all_feats.append(feats.numpy())
            all_coords.append(np.asarray(valid_rows, dtype=np.int64))
            del images, feats
    finally:
        def _close_on_worker(ctx):
            ctx.__exit__(None, None, None)
        reader_pool.submit(_close_on_worker, czi_ctx).result()

    if not all_feats:
        log(f"  [WARN] {slide_stem}: no patches were successfully read/encoded, skipping.")
        write_checkpoint("slide_skip_no_patches", slide=slide_stem)
        return None

    features_np = np.concatenate(all_feats, axis=0)            # (N, 768)
    patch_xy    = np.concatenate(all_coords, axis=0)           # (N, 2) in patch_x/patch_y units

    if COORD_MODE == "level0":
        coords = patch_xy.astype(np.int64) * downsample        # true level-0 pixel coords
        patch_size_lv0 = PATCH_SIZE * downsample                # e.g. 1024 on a 40x slide
    else:  # "legacy": reproduce Titan_Project.ipynb's halved-coord bug for comparison
        coords = patch_xy.astype(np.int64)
        patch_size_lv0 = PATCH_SIZE * downsample

    # Guard the exact bug this notebook fixes: every patch must land in its own
    # grid cell once TITAN floor-divides coords by patch_size_lv0.
    grid = (coords - coords.min(axis=0)) // patch_size_lv0
    n_unique = len(np.unique(grid, axis=0))
    if n_unique != len(grid):
        raise AssertionError(
            f"{slide_stem}: grid collapse detected ({len(grid)} patches -> "
            f"{n_unique} unique grid cells). coords/patch_size_lv0 mismatch "
            f"(patch_size_lv0={patch_size_lv0}, downsample={downsample}, COORD_MODE={COORD_MODE})."
        )

    device = extractor.device
    features_torch = torch.from_numpy(features_np).unsqueeze(0).to(device)
    coords_torch   = torch.from_numpy(coords).unsqueeze(0).to(device)

    with torch.inference_mode():
        slide_emb = extractor.titan.encode_slide_from_patch_features(
            features_torch, coords_torch, patch_size_lv0
        )

    out_path = os.path.join(OUTPUT_DIR, f"{slide_stem}.pt")
    _atomic_save(slide_emb.cpu(), out_path)

    elapsed = time.time() - t0
    log(f"  -> {slide_stem}: {len(rows)} patches, embedding {tuple(slide_emb.shape)} "
        f"saved to {out_path} ({elapsed:.1f}s)")
    write_checkpoint("slide_done", slide=slide_stem, n_patches=len(rows), elapsed_sec=round(elapsed, 1))
    return out_path


In [ ]:
# =============================================================================
# ORCHESTRATOR
# =============================================================================

def find_czi_on_disk(root: str) -> dict:
    # stem -> full path, for every .czi currently sitting in CZI_DIR.
    return {Path(p).stem: p for p in glob.glob(os.path.join(root, "*.czi"))}


def main():
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")

    log("=" * 70)
    log("  TITAN SurGen CZI pipeline")
    log("=" * 70)
    if torch.cuda.is_available():
        log(f"  CUDA device: {torch.cuda.get_device_name(0)}")

    all_wsis = build_wsi_list(SURGEN_LABELS_CSV)
    batches = [all_wsis[i:i + BATCH_SIZE] for i in range(0, len(all_wsis), BATCH_SIZE)]
    log(f"  Total valid WSIs : {len(all_wsis)}  |  Batches: {len(batches)}")
    log(f"  Download enabled : {DOWNLOAD_ENABLED}  |  Delete after: {DELETE_AFTER}")

    # -- Resume scan: jump to the first batch that isn't fully complete --------
    start_batch_idx = len(batches)
    for i, b in enumerate(batches):
        if not _batch_fully_complete(b):
            start_batch_idx = i
            break
    log(f"  [RESUME SCAN] Skipping {start_batch_idx} already-complete batch(es). "
        f"Starting at batch {start_batch_idx + 1}/{len(batches)}.")

    extractor = None
    try:
        for batch_idx, batch in enumerate(batches[start_batch_idx:], start=start_batch_idx):
            log(f"\n{'-'*70}\nBATCH {batch_idx + 1}/{len(batches)}  ({len(batch)} WSI(s))\n{'-'*70}")
            write_checkpoint("batch_start", batch_idx=batch_idx, n_wsis=len(batch))

            if _batch_fully_complete(batch):
                log("  [SKIP BATCH] Already fully processed.")
                continue

            if DOWNLOAD_ENABLED:
                to_download = filter_batch_for_download(batch)
                if to_download:
                    download_batch(to_download)
            else:
                log("  [DOWNLOAD DISABLED] Only processing CZIs already present in CZI_DIR.")

            czi_index = find_czi_on_disk(CZI_DIR)

            if extractor is None:
                extractor = TITANExtractor()

            for filename, _url in batch:
                stem = Path(filename).stem
                if slide_ready_to_skip(stem):
                    continue
                if stem not in czi_index:
                    log(f"  [SKIP] {stem}: CZI not found on disk (download disabled or failed).")
                    continue

                try:
                    encode_slide(stem, czi_index[stem], extractor, NONWHITE_INDEX[stem])
                except Exception as e:
                    log_exception(f"encode_slide stem={stem}", e)
                    log(f"  [ERROR] {stem} failed: {e}")

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            if DELETE_AFTER:
                delete_batch_czis(batch)

            sync_to_remote(label=f"(batch {batch_idx + 1}/{len(batches)})")

            if DOWNLOAD_ENABLED:
                check_disk_space_or_halt(MIN_FREE_GB)

            write_checkpoint("batch_complete", batch_idx=batch_idx)
    finally:
        if extractor is not None:
            del extractor
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    log("\nALL BATCHES COMPLETE")
    log(f"  Output dir: {OUTPUT_DIR}")
    write_checkpoint("all_batches_complete")


if __name__ == "__main__":
    try:
        main()
    except SystemExit:
        raise
    except BaseException as e:
        log_exception("main_top_level", e)
        write_checkpoint("main_crashed_python_exception", error=str(e))
        raise


## Verification

Inspect one produced embedding: shape, dtype, and basic sanity stats.

In [ ]:
def analyze_pt_file(pt_path: str):
    data = torch.load(pt_path, map_location="cpu")
    if isinstance(data, dict):
        data = list(data.values())[0]
    print(f"File   : {pt_path}")
    print(f"Shape  : {tuple(data.shape)}")
    print(f"Dtype  : {data.dtype}  Device: {data.device}")
    flat = data.flatten().float()
    print(f"Min/Max: {flat.min().item():.4f} / {flat.max().item():.4f}")
    print(f"Mean/Std: {flat.mean().item():.4f} / {flat.std().item():.4f}")
    print(f"NaN/Inf: {torch.isnan(flat).sum().item()} / {torch.isinf(flat).sum().item()}")
    print(f"L2 norm: {flat.norm().item():.4f}")


# Example - point this at any slide produced above:
# analyze_pt_file(os.path.join(OUTPUT_DIR, "SR1482_40X_HE_T390_01.pt"))

_existing = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.pt")))
print(f"{len(_existing)} embedding(s) currently in {OUTPUT_DIR}")
if _existing:
    analyze_pt_file(_existing[0])
